In [2]:
import pandas as pd
import datetime as dt
df = pd.read_parquet(r'c:\Users\barra\Documents\VS CODE\nyc_taxi_data_pipeline\data\raw\yellow_tripdata_2026-01.parquet')
missing = df.passenger_count.isna()
df.head(20)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0.0,2.90,1.0,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


In [8]:
display(missing)
print(missing.sum())
df.loc[missing, "VendorID"].value_counts()
missingness = df.groupby('VendorID').passenger_count.apply(lambda x:x.isna().mean() * 100)
print(missingness)
print(df.trip_distance.describe())
over_miles = df.trip_distance > 100
print(over_miles.sum())
print(162 / df.trip_distance.notna().sum()*100)
print(df.trip_distance.nlargest(10))
# print(df.loc[df['trip_distance'] > 100, ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'fare_amount', 'total_amount']].head(10))
duration = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
display(duration[df['trip_distance'] > 100].dt.total_seconds()/60)
print(((df.trip_distance > 100) & (duration.dt.total_seconds() / 60 < 100)).sum())
display(df.loc[df.trip_distance > 100, ['trip_distance', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'fare_amount', 'total_amount']].sort_values('trip_distance', ascending=False))
print(((df.trip_distance > 100) & (df.fare_amount > 100)).sum())
print(df.trip_distance.quantile([0.90, 0.95, 0.99, 0.999]))
print((df.trip_distance == 0).sum())
display(df.loc[df.trip_distance == 0, ['trip_distance', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'fare_amount', 'total_amount']])
print(((df.trip_distance ==0) & (df.fare_amount < 0)).sum())
print(((df.trip_distance ==0) & (df.fare_amount == 0)).sum())
print(duration[df.trip_distance == 0].dt.total_seconds().describe())
print(duration[df.trip_distance == 0].nlargest(10))


display(df)
display(df.loc[1560244])
display(df.loc[[572461, 100303, 469655, 1445009, 3166144, 1185418, 145446, 145447]])
display(df.loc[(duration > pd.Timedelta(hours=24)) &( df.trip_distance == 0)]) 

0          False
1          False
2          False
3          False
4          False
           ...  
3724884     True
3724885     True
3724886     True
3724887     True
3724888     True
Name: passenger_count, Length: 3724889, dtype: bool

1088058
VendorID
1     19.622902
2     31.851557
6    100.000000
7      0.000000
Name: passenger_count, dtype: float64
count    3.724889e+06
mean     6.455647e+00
std      6.488855e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.810000e+00
75%      3.730000e+00
max      2.690975e+05
Name: trip_distance, dtype: float64
162
0.0043491228866148765
3524923    269097.48
3632367    237567.19
3285750    236216.69
3131676    236183.06
2704619    221544.91
2688274    221436.73
3131406    196490.48
3571687    192081.68
2820951    184908.33
3499856    183363.75
Name: trip_distance, dtype: float64


77980       15.350000
191169      11.816667
252325     137.400000
266099     134.133333
290678     118.333333
              ...    
3630323    168.483333
3632367     38.000000
3634297     32.000000
3661041      4.000000
3687462    122.716667
Length: 162, dtype: float64

120


,trip_distance,tpep_pickup_datetime,tpep_dropoff_datetime,fare_amount,total_amount
3524923,269097.48,2026-01-28 01:08:00,2026-01-28 01:16:00,12.19,16.94
3632367,237567.19,2026-01-30 13:00:00,2026-01-30 13:38:00,72.99,81.95
3285750,236216.69,2026-01-21 16:31:00,2026-01-21 16:40:00,13.58,21.33
3131676,236183.06,2026-01-17 10:40:00,2026-01-17 10:57:00,22.75,34.96
2704619,221544.91,2026-01-02 21:31:00,2026-01-02 21:55:00,31.21,39.55
...,...,...,...,...,...
1508035,103.64,2026-01-18 11:49:04,2026-01-18 13:38:06,628.80,663.26
1646940,101.90,2026-01-20 09:15:07,2026-01-20 09:35:19,19.50,20.00
2620130,101.65,2026-01-31 20:35:07,2026-01-31 22:29:40,630.90,680.61
2620129,101.65,2026-01-31 20:35:07,2026-01-31 22:29:40,-630.90,-680.61


...

2755
501
count    125738.000000
mean        932.221150
std        2584.544944
min           0.000000
25%         271.000000
50%         798.000000
75%        1267.000000
max      450474.000000
dtype: float64
1560244   5 days 05:07:54
572461    4 days 03:14:14
100303    3 days 06:32:30
469655    2 days 14:30:02
1445009   1 days 20:48:23
3166144   1 days 09:22:49
1185418   1 days 00:22:34
1154357   1 days 00:19:26
145446    0 days 23:58:26
145447    0 days 23:58:26
dtype: timedelta64[us]


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,NaN,237,161,0,17.09,0.00,0.5,0.00,0.0,1.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,NaN,42,42,0,23.19,0.00,0.5,0.00,0.0,1.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,NaN,137,69,0,29.21,0.00,0.5,0.00,0.0,1.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,NaN,137,162,0,19.19,0.00,0.5,0.00,0.0,1.0,23.94,NaN,NaN,0.75


VendorID                                   2
tpep_pickup_datetime     2026-01-19 00:35:57
tpep_dropoff_datetime    2026-01-24 05:43:51
passenger_count                          1.0
trip_distance                            0.0
RatecodeID                               2.0
store_and_fwd_flag                         N
PULocationID                             132
DOLocationID                             132
payment_type                               2
fare_amount                             70.0
extra                                    0.0
mta_tax                                  0.5
tip_amount                               0.0
tolls_amount                             0.0
improvement_surcharge                    1.0
total_amount                            71.5
congestion_surcharge                     0.0
Airport_fee                              0.0
cbd_congestion_fee                       0.0
Name: 1560244, dtype: object

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
572461,2,2026-01-08 12:06:24,2026-01-12 15:20:38,1.0,0.0,1.0,N,207,132,2,3.70,0.0,0.5,0.0,0.0,1.0,5.20,0.0,0.0,0.00
100303,1,2026-01-02 14:11:42,2026-01-05 20:44:12,1.0,0.0,99.0,N,181,181,1,25.50,0.0,0.5,0.0,0.0,0.0,26.00,0.0,0.0,0.00
469655,1,2026-01-07 10:56:28,2026-01-10 01:26:30,1.0,0.0,99.0,N,71,71,1,40.50,0.0,0.5,0.0,0.0,0.0,41.00,0.0,0.0,0.00
1445009,2,2026-01-17 16:03:57,2026-01-19 12:52:20,1.0,0.0,1.0,N,19,193,2,7.20,0.0,0.5,0.0,0.0,1.0,8.70,0.0,0.0,0.00
3166144,2,2026-01-18 01:24:22,2026-01-19 10:47:11,NaN,0.0,NaN,NaN,45,193,0,32.26,0.0,0.5,0.0,0.0,1.0,37.01,NaN,NaN,0.75
1185418,1,2026-01-14 21:35:30,2026-01-15 21:58:04,1.0,0.0,99.0,N,168,168,1,43.50,0.0,0.5,0.0,0.0,0.0,44.00,0.0,0.0,0.00
145446,2,2026-01-03 01:23:38,2026-01-04 01:22:04,0.0,0.0,5.0,N,79,79,2,0.00,0.0,-0.5,0.0,0.0,-1.0,-4.75,-2.5,0.0,-0.75
145447,2,2026-01-03 01:23:38,2026-01-04 01:22:04,0.0,0.0,5.0,N,79,79,2,0.00,0.0,0.5,0.0,0.0,1.0,4.75,2.5,0.0,0.75


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
100303,1,2026-01-02 14:11:42,2026-01-05 20:44:12,1.0,0.0,99.0,N,181,181,1,25.50,0.0,0.5,0.0,0.0,0.0,26.00,0.0,0.0,0.00
469655,1,2026-01-07 10:56:28,2026-01-10 01:26:30,1.0,0.0,99.0,N,71,71,1,40.50,0.0,0.5,0.0,0.0,0.0,41.00,0.0,0.0,0.00
572461,2,2026-01-08 12:06:24,2026-01-12 15:20:38,1.0,0.0,1.0,N,207,132,2,3.70,0.0,0.5,0.0,0.0,1.0,5.20,0.0,0.0,0.00
1154357,2,2026-01-14 17:08:07,2026-01-15 17:27:33,1.0,0.0,99.0,N,162,14,1,22.60,0.0,0.5,0.0,0.0,1.0,27.35,2.5,0.0,0.75
1185418,1,2026-01-14 21:35:30,2026-01-15 21:58:04,1.0,0.0,99.0,N,168,168,1,43.50,0.0,0.5,0.0,0.0,0.0,44.00,0.0,0.0,0.00
1445009,2,2026-01-17 16:03:57,2026-01-19 12:52:20,1.0,0.0,1.0,N,19,193,2,7.20,0.0,0.5,0.0,0.0,1.0,8.70,0.0,0.0,0.00
1560244,2,2026-01-19 00:35:57,2026-01-24 05:43:51,1.0,0.0,2.0,N,132,132,2,70.00,0.0,0.5,0.0,0.0,1.0,71.50,0.0,0.0,0.00
3166144,2,2026-01-18 01:24:22,2026-01-19 10:47:11,NaN,0.0,NaN,NaN,45,193,0,32.26,0.0,0.5,0.0,0.0,1.0,37.01,NaN,NaN,0.75
